## Konversi SEG2 ke MiniSEED
Author: Annora Vandanu Erlangga

#### 1. Import library
Library yang digunakan adalah obspy, os, dan glob
- Menggunakan `obspy` untuk membaca file `.seg2` dan menyimpan ke format `.mseed`
- Menggunakan `glob.glob()` untuk mencari seluruh file `.seg2` dalam folder target

In [126]:
from obspy.core import read, Stream
import os
import glob

In [127]:
folder_path = r'D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7'
files = glob.glob(os.path.join(folder_path, '*.seg2'))

In [128]:
print(files)

['D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104228299.WIT.3c.tst.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104242000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104342000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104442000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104542000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104642000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104742000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulon Progo\\Kelompok 6_Day 3\\B7\\20250412_104842000.WIT.3c.cont.0.seg2', 'D:\\Kulon Progo 2025\\Seismik Pasif\\data\\Kulo

#### 2. Pembersihan file
Menghapus file pertama dan terakhir dalam list karena file pertama adalah testing/kalibrasi dari alat

In [129]:
del files[0]
del files[-1]
data=[None]*len(files)

#### 3. Membaca dan Menggabungkan Data
- Setiap file `.seg2` memiliki 3 trace (komponen).
- Menggabungkan trace dari seluruh file untuk masing-masing komponen:
    - `x_comp` untuk timur-barat (EHE)
    - `y_comp` untuk utara-selatan (EHN)
    - `z_comp` untuk vertikal (EHZ)

[Cek manual book SUMMIT M VIPA]

In [130]:
# DEBUG error files --------- B6 station

for file in files:
    try:
        st = read(file)
        print(f"{file} OK")
    except Exception as e:
        print(f"{file} ERROR: {e}")


D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104242000.WIT.3c.cont.0.seg2 OK


D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104342000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104442000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104542000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104642000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104742000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104842000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_104942000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_105042000.WIT.3c.cont.0.seg2 OK
D:\Kulon Progo 2025\Seismik Pasif\data\Kulon Progo\Kelompok 6_Day 3\B7\20250412_105142000.WIT.3c.cont.0.

In [131]:
for i in range(len(files)):
    data[i]=read(files[i])

x_comp=data[0].traces[0]
y_comp=data[0].traces[1]
z_comp=data[0].traces[2]

for j in range(1,len(files)):
    x_comp+=data[j].traces[0]
    y_comp+=data[j].traces[1]
    z_comp+=data[j].traces[2]

#### 4. Membuat Stream Gabungan
Ketiga komponen digabung jadi satu `Stream` lalu disortir berdasarkan waktu

In [132]:
x_comp.stats.channel = "EHE"  # Komponen timur-barat
y_comp.stats.channel = "EHN"  # Komponen utara-selatan
z_comp.stats.channel = "EHZ"  # Komponen vertikal

# Gabungkan ketiga komponen menjadi satu Stream
combined_stream = Stream(traces=[x_comp, y_comp, z_comp])

#### 5. Output
File hasil akhir adalah 1 buah file `.mseed` yang berisi 3 komponen dari seluruh data `.seg2` dalam folder

In [133]:
# Ambil nama folder terakhir dari path sebagai nama file
folder_name = os.path.basename(folder_path)

output_filename = f"{folder_name}_merge3comps.mseed"
output = r"D:\Kulon Progo 2025\Seismik Pasif\processing\1_merge mseed\Merged data"
output_path = os.path.join(output, output_filename)

# Simpan stream gabungan dalam format MiniSEED
combined_stream.write(output_path, format='MSEED')
print(f"File MSEED gabungan telah disimpan di: {output_path}")

File MSEED gabungan telah disimpan di: D:\Kulon Progo 2025\Seismik Pasif\processing\1_merge mseed\Merged data\B7_merge3comps.mseed
